## STEP 0 — Stop vLLM (clean slate)

In [1]:
!fuser -k 8000/tcp || true
!pkill -f vllm || true

^C


In [2]:
!ss -ltnp | grep ':8000' || echo "vLLM stopped"

vLLM stopped


## STEP 1 - Install SGLang

In [ ]:
!pip install -U sglang

## STEP 2 - Start SGLang server

In [3]:
!nohup python -m sglang.launch_server \
  --model TinyLlama/TinyLlama-1.1B-Chat-v1.0 \
  --host 0.0.0.0 \
  --port 30000 > sglang.log 2>&1 &

## STEP 3 - Verify SGLang is running

### Check Logs

In [5]:
!tail -n 40 sglang.log

W0000 00:00:1765778413.230414    8412 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1765778413.230415    8413 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1765778413.230417    8413 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
[2025-12-15 06:00:17] INFO utils.py:164: NumExpr defaulting to 12 threads.
[2025-12-15 06:00:17] INFO utils.py:164: NumExpr defaulting to 12 threads.
[2025-12-15 06:00:24] Init torch distributed begin.
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0


### Health check

In [6]:
!curl --max-time 5 -s -o /dev/null -w "%{http_code}\n" http://127.0.0.1:30000/health

200


## STEP 4 - Start GPU utilization logging

In [7]:
!nohup bash -lc 'nvidia-smi --query-gpu=timestamp,utilization.gpu,utilization.memory,memory.used --format=csv -l 1 > gpu_util_sglang.csv' >/dev/null 2>&1 &
!echo "GPU logging started for SGLang"

GPU logging started for SGLang


## Step 5 - Run SGLang Benchmark

In [8]:
import time, requests, pandas as pd

URL = "http://127.0.0.1:30000/v1/chat/completions"
MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

PROMPTS = [
    "Explain cloud computing in one sentence.",
    "Summarize: model serving platforms need to balance latency, throughput, and cost.",
    "Write 3 bullet points comparing CPU vs GPU for inference.",
    "Explain what batching means in inference and why it improves throughput.",
]

def run_once(prompt, max_tokens=128, temperature=0.0):
    payload = {
        "model": MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "temperature": temperature,
    }
    t0 = time.time()
    r = requests.post(URL, json=payload, timeout=180)
    t1 = time.time()
    r.raise_for_status()
    data = r.json()
    text = data["choices"][0]["message"]["content"]
    usage = data.get("usage", {})
    return {
        "latency_s": t1 - t0,
        "output_text": text,
        "prompt_tokens": usage.get("prompt_tokens", None),
        "completion_tokens": usage.get("completion_tokens", None),
    }

rows = []

# Warm-up
_ = run_once("Say hello.", max_tokens=32)

for max_toks in [64, 128, 256]:
    for prompt in PROMPTS:
        for trial in range(5):
            out = run_once(prompt, max_tokens=max_toks)
            approx_tokens = len(out["output_text"].split())
            gen_tokens = out["completion_tokens"] if out["completion_tokens"] else approx_tokens
            rows.append({
                "framework": "SGLang",
                "model": MODEL,
                "max_tokens": max_toks,
                "prompt_len_chars": len(prompt),
                "trial": trial,
                "latency_s": out["latency_s"],
                "gen_tokens": gen_tokens,
                "tokens_per_sec": gen_tokens / out["latency_s"],
            })

df_sglang = pd.DataFrame(rows)
df_sglang.head()

,framework,model,max_tokens,prompt_len_chars,trial,latency_s,gen_tokens,tokens_per_sec
0,SGLang,TinyLlama/TinyLlama-1.1B-Chat-v1.0,64,40,0,8.431182,28,3.321005
1,SGLang,TinyLlama/TinyLlama-1.1B-Chat-v1.0,64,40,1,0.105555,28,265.265608
2,SGLang,TinyLlama/TinyLlama-1.1B-Chat-v1.0,64,40,2,0.095602,28,292.880063
3,SGLang,TinyLlama/TinyLlama-1.1B-Chat-v1.0,64,40,3,0.096549,28,290.007364
4,SGLang,TinyLlama/TinyLlama-1.1B-Chat-v1.0,64,40,4,0.097439,28,287.358418


## Step 6 - Stop GPU logging

In [9]:
!pkill -f "gpu_util_sglang.csv" || true
!echo "GPU logging stopped"

^C
GPU logging stopped


## Step 7 - Save SGLang results

In [10]:
df_sglang.to_csv("sglang_tinyllama_results.csv", index=False)
print("Saved sglang_tinyllama_results.csv")

Saved sglang_tinyllama_results.csv


## STEP 8 - Quick sanity checks

In [11]:
!head gpu_util_sglang.csv
!tail gpu_util_sglang.csv

timestamp, utilization.gpu [%], utilization.memory [%], memory.used [MiB]
2025/12/15 06:02:10.710, 0 %, 0 %, 69273 MiB
2025/12/15 06:02:11.713, 0 %, 0 %, 69273 MiB
2025/12/15 06:02:12.715, 0 %, 0 %, 69273 MiB
2025/12/15 06:02:13.716, 0 %, 0 %, 69273 MiB
2025/12/15 06:02:14.718, 0 %, 0 %, 69273 MiB
2025/12/15 06:02:15.720, 0 %, 0 %, 69273 MiB
2025/12/15 06:02:16.722, 0 %, 0 %, 69273 MiB
2025/12/15 06:02:17.724, 0 %, 0 %, 69273 MiB
2025/12/15 06:02:18.726, 0 %, 0 %, 69273 MiB
2025/12/15 06:03:11.818, 0 %, 0 %, 69273 MiB
2025/12/15 06:03:12.820, 0 %, 0 %, 69273 MiB
2025/12/15 06:03:13.822, 0 %, 0 %, 69273 MiB
2025/12/15 06:03:14.822, 0 %, 0 %, 69273 MiB
2025/12/15 06:03:15.824, 0 %, 0 %, 69273 MiB
2025/12/15 06:03:16.826, 0 %, 0 %, 69273 MiB
2025/12/15 06:03:17.828, 0 %, 0 %, 69273 MiB
2025/12/15 06:03:18.829, 0 %, 0 %, 69273 MiB
2025/12/15 06:03:19.831, 0 %, 0 %, 69273 MiB
2025/12/15 06:03:20.833, 0 %, 0 %, 69273 MiB
